## Score calculation of different models

In [2]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASETS = ["CaseReportBench", "PHEE", "DiscourseEE", "MACCROBAT"]
QG_Models = ["gpt-oss-120b", "qwen3-4b", "qwen3-8b"]
PD_Models = ["gpt-oss-120b", "qwen3-4b", "qwen3-8b"]

OUTPUTS_SC_DIR = "/dartfs/rc/home/j/f006f3j/lab/omar/LoQA/Outputs/sc"

SETUPS = ["schema", "cot-schema", "loqa", "optimized_loqa"]
MODELS = PD_Models


In [3]:
from pathlib import Path


def load_scores(dataset: str, setup: str, model: str):
    """Return (relaxed_f1, complex_f1) for a given config, or (None, None) if missing."""
    base_dir = Path(OUTPUTS_SC_DIR) / dataset

    if setup in {"schema", "cot-schema"}:
        filename = f"{setup}-{model}-zs-v0-{dataset}-gold-test-{model}-zs-v0.json"
    else:  # loqa / optimized_loqa: QG model is fixed to gpt-oss-120b
        qg_model = "gpt-oss-120b"
        filename = f"{setup}-{qg_model}-zs-v0-{dataset}-gold-test-{model}-zs-v0.json"

    path = base_dir / filename
    if not path.exists():
        return None, None

    with path.open("r") as f:
        data = json.load(f).get("overall", {})

    relaxed_f1 = data.get("relaxed-match-f1")
    complex_f1 = data.get("complex-match-f1")
    return relaxed_f1, complex_f1


for dataset in DATASETS:
    rows = []
    for setup in SETUPS:
        row = {"setup": setup}
        for model in MODELS:
            relaxed, complex_ = load_scores(dataset, setup, model)
            row[(model, "relaxed-match-f1")] = relaxed
            row[(model, "complex-match-f1")] = complex_
        rows.append(row)

    df = pd.DataFrame(rows).set_index("setup")

    # Convert dict-style columns to a proper MultiIndex: (model, metric)
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["model", "metric"])

    print("-" * 100)
    print(dataset)
    display(df.round(2))


----------------------------------------------------------------------------------------------------
CaseReportBench


model              gpt-oss-120b                          qwen3-4b  \
metric         relaxed-match-f1 complex-match-f1 relaxed-match-f1   
setup                                                               
schema                    21.18            57.45            17.18   
cot-schema                17.41            54.80            17.88   
loqa                      37.31            78.26            28.30   
optimized_loqa            38.91            84.86            28.75   

model                                   qwen3-8b                   
metric         complex-match-f1 relaxed-match-f1 complex-match-f1  
setup                                                              
schema                    45.21            13.36            39.20  
cot-schema                44.35            14.04            38.73  
loqa                      59.15            24.49            54.05  
optimized_loqa            62.11            26.25            59.35

----------------------------------------------------------------------------------------------------
PHEE


model              gpt-oss-120b                          qwen3-4b  \
metric         relaxed-match-f1 complex-match-f1 relaxed-match-f1   
setup                                                               
schema                    49.61            67.84            57.51   
cot-schema                50.62            71.25            52.78   
loqa                      67.69            84.40            62.50   
optimized_loqa            76.27            91.97            64.82   

model                                   qwen3-8b                   
metric         complex-match-f1 relaxed-match-f1 complex-match-f1  
setup                                                              
schema                    73.09            62.53            78.50  
cot-schema                65.19            59.08            71.91  
loqa                      78.69            65.90            80.82  
optimized_loqa            80.19            68.84            83.03

----------------------------------------------------------------------------------------------------
DiscourseEE


model              gpt-oss-120b                          qwen3-4b  \
metric         relaxed-match-f1 complex-match-f1 relaxed-match-f1   
setup                                                               
schema                    13.12            48.83            14.37   
cot-schema                11.56            51.61            14.57   
loqa                      20.10            63.02            19.01   
optimized_loqa            24.39            75.76            20.24   

model                                   qwen3-8b                   
metric         complex-match-f1 relaxed-match-f1 complex-match-f1  
setup                                                              
schema                    46.93            16.09            48.63  
cot-schema                46.09            16.46            46.63  
loqa                      55.97            20.89            54.51  
optimized_loqa            59.12            22.22            57.27

----------------------------------------------------------------------------------------------------
MACCROBAT


model              gpt-oss-120b                          qwen3-4b  \
metric         relaxed-match-f1 complex-match-f1 relaxed-match-f1   
setup                                                               
schema                    20.99            41.85            21.54   
cot-schema                20.95            41.63            23.39   
loqa                      56.79            75.78            40.03   
optimized_loqa            72.88            91.67            49.78   

model                                   qwen3-8b                   
metric         complex-match-f1 relaxed-match-f1 complex-match-f1  
setup                                                              
schema                    37.87            25.61            39.58  
cot-schema                37.83            30.66            41.04  
loqa                      58.39            45.09            60.86  
optimized_loqa            65.59            57.08            72.69